Paper Pipeline Protoyping

In [55]:
# psql -h localhost -d research_platform -U postgres -f platform_schema.sql

In [56]:
import torch
print(f"PyTorch version: {torch.__version__}")
torch.cuda.is_available()

PyTorch version: 2.7.0


True

In [57]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_postgres import PGVector
import pandas as pd
from sqlalchemy import create_engine
import json
import psycopg
import ray
import dask.dataframe as dd
import dask
import modin.pandas as mdf
import os
import polars as pl
from pgvector.sqlalchemy import Vector
from dotenv import load_dotenv
load_dotenv()
ray.init(num_gpus=1, ignore_reinit_error=True)

2025-06-09 21:25:01,487	INFO worker.py:1718 -- Calling ray.init() again after it has already been called.


Python version:,3.12.10
Ray version:,2.46.0


In [58]:
# list the available CPU cores
total_cores = ray.cluster_resources()['CPU']
print(f"Total CPU cores available: {total_cores}")
ray.available_resources()


Total CPU cores available: 128.0


{'node:192.168.50.241': 1.0,
 'accelerator_type:G': 1.0,
 'node:__internal_head__': 1.0,
 'CPU': 128.0,
 'memory': 76222264116.0,
 'object_store_memory': 32666684620.0,
 'GPU': 1.0}

In [59]:
def create_embedding(model, sentences):
  """
  Create embeddings for a list of sentences using the specified model.
  
  Args:
      model: The model to use for creating embeddings.
      sentences: A list of sentences to embed.
  
  Returns:
      A list of embeddings for the input sentences.
  """
  embeddings = model.encode(sentences, convert_to_tensor=True)
  return embeddings

In [60]:
arxiv_bert_model = SentenceTransformer('ArtifactAI/arxiv-distilroberta-base-GenQ', device='cuda' if torch.cuda.is_available() else 'cpu')
# output_embeddings = create_embedding(
#   model=arix_bert_model,
#   sentences=["This is a test sentence", "This is another test sentence"]
# )

In [61]:
@ray.remote(num_gpus=1)
def process_paper_metadata_embeddings(model,dataframe):
  """
  Process a JSON file containing paper metadata and extract relevant information.
  
  """
  dataframe["embedding"] = dataframe.apply(
    lambda row: create_embedding(
      model,
      row['abstract'],
    ).tolist(),
    axis=1
  )
  dataframe = dataframe.rename(columns={'abstract': 'context'})
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe[["paper_id", "embedding", "context"]]
  
  return dataframe


In [62]:
async def store_row_in_postgres(connection, row):
  """
  Store a row of data in the PostgreSQL database.
  
  Args:
      connection: The database connection object.
      row: The row of data to store.
  
  """
  paper_id = row['paper_id']
  embedding = row['embedding']
  context = row['context']
  
  # Insert the data into the database
  await connection.execute(
    "INSERT INTO paper_vector_db (paper_id, embedding, context) VALUES (%s, %s, %s)",
    (paper_id, embedding, context),
  )
  await connection.commit()

In [63]:
connection = await psycopg.AsyncConnection.connect(
  os.environ['SELF_HOSTED_POSTGRES_CONNECTION']
)

### Parallelizing paper upload



Use Modin Dataframes for parallel processing, wraps pandas in ray context


In [64]:
def fetch_computer_science_paper_metadata(dataframe):
  # 
  computer_science_paper_metadata_mdf = dataframe[
    dataframe['categories'].fillna('').str.contains('cs.', case=False, na=False)
  ]
  return computer_science_paper_metadata_mdf
  

In [65]:
def process_paper_metadata_papers(dataframe):
#  Index(['paper_id', 'submitter', 'authors', 'title', 'comments', 'journal-ref',
#        'doi', 'report-no', 'categories', 'license', 'abstract', 'versions',
#        'update_date', 'authors_parsed'],
#       dtype='object')
  print("Initial keys",dataframe.keys())
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe.rename(columns={'report-no': 'report_number'})
  dataframe = dataframe.fillna('').astype(str)
  print("Output keys",dataframe.keys())
  dataframe = dataframe[["paper_id", "title", "abstract", "doi", "report_number"]]
  return dataframe 
  

In [66]:
def process_paper_document_metadata(dataframe):
  
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe.rename(columns={'title': 'file_name'})
  dataframe["owner_id"] = "public"
  dataframe["group_id"] = "public"
  dataframe["url"] = dataframe.apply(lambda row: f"/arxiv.org/pdf/{row['paper_id']}", axis=1)
  print(dataframe.keys())
  return dataframe[["paper_id", "owner_id", "group_id", "file_name", "url"]]



In [67]:
paper_metadata_df = pd.read_json('arxiv-metadata-oai-snapshot.json', lines=True).head(10)

In [68]:
display(paper_metadata_df.head())

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [69]:
from sqlalchemy import text

# add vector support to the engine


engine = create_engine(os.environ["SELF_HOSTED_POSTGRES_CONNECTION"])


alchemy_connection = engine.connect()
alchemy_connection.execute(text('CREATE EXTENSION IF NOT EXISTS vector'))


In [70]:
cs_paper_df = fetch_computer_science_paper_metadata(paper_metadata_df)
cs_paper_mdf = mdf.DataFrame(cs_paper_df)


paper vector db upload


In [79]:
embedding_mdf = ray.get(process_paper_metadata_embeddings.remote(arxiv_bert_model, paper_metadata_df))
async def insert_embeddings(connection, embedding_mdf):
  records = [
    (row['paper_id'], row['embedding'], row['context'])
    for _, row in embedding_mdf.iterrows()
  ]
  async with connection.cursor() as cur:
    await cur.executemany(
      "INSERT INTO paper_vector_db (paper_id, embedding, context) VALUES (%s, %s, %s)",
      records
    )
    await connection.commit()

In [80]:
await insert_embeddings(connection=connection,embedding_mdf=embedding_mdf)

In [ ]:
process_paper_metadata_papers(cs_paper_df.copy()).to_sql('paper_metadata',alchemy_connection, if_exists='replace', index=False)

Initial keys Index(['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi',
       'report-no', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')
Output keys Index(['paper_id', 'submitter', 'authors', 'title', 'comments', 'journal-ref',
       'doi', 'report_number', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')


2

In [ ]:
display(cs_paper_mdf.keys())

Index(['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi',
       'report-no', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')

In [ ]:
process_paper_document_metadata(cs_paper_df.copy()).to_sql("documents", con=alchemy_connection, if_exists="replace", index=False)

Index(['paper_id', 'submitter', 'authors', 'file_name', 'comments',
       'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract',
       'versions', 'update_date', 'authors_parsed', 'owner_id', 'group_id',
       'url'],
      dtype='object')


2

In [ ]:
display(cs_paper_mdf.head())

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"


In [ ]:
ray.shutdown()

I0000 00:00:1749528091.277336   48089 chttp2_transport.cc:1182] ipv4:192.168.50.241:60342: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {grpc_status:14, http2_error:2, created_time:"2025-06-09T21:01:31.27733162-07:00"}


In [ ]:
sync_connection = psycopg.connect(
  os.environ['SELF_HOSTED_POSTGRES_CONNECTION']
)
result =sync_connection.execute(
  "SELECT * FROM paper_vector_db;"
)
result_dict = result.fetchall()

In [ ]:
display(result_dict[0][1])

'{-0.02221376821398735,-0.02284185215830803,0.017939966171979904,-0.5473503470420837,-0.17219851911067963,-0.14693816006183624,0.02166121080517769,0.03382785990834236,0.05365683510899544,-0.06282669305801392,0.35840752720832825,-0.16424213349819183,0.28802913427352905,-0.3820122480392456,0.017693262547254562,-0.18410398066043854,0.2947666049003601,-0.7180113792419434,0.07303538918495178,0.46925801038742065,0.48016849160194397,-1.0009331703186035,0.10231275111436844,-0.3701360821723938,0.6909980177879333,0.11818594485521317,-0.19581782817840576,-0.054143037647008896,0.5461544990539551,-1.0881317853927612,0.2266409695148468,0.028451768681406975,-0.0926627367734909,-0.1781274825334549,-0.18542271852493286,-0.15120965242385864,-0.16811080276966095,0.27875813841819763,-0.5428684949874878,-0.26670345664024353,0.31100139021873474,-0.7332471609115601,0.14510445296764374,-0.22941753268241882,-0.5733129382133484,-0.14508147537708282,-0.27356573939323425,0.09667808562517166,0.5831335186958313,-0.

In [ ]:
sync_connection.close()

In [ ]:
await connection.close()